In [2]:
import sys
sys.path.insert(0, "..")

from src.retrieval import search_hybrid, chunks_df

print("chunks:", len(chunks_df))
print(search_hybrid("minimum capital for a commercial bank")[0]["title"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

chunks: 3828
Review of Minimum Capital Requirements for Commercial, Merchant and Non-Interest Banks in Nigeria


In [3]:
from src.retrieval import collection
assert collection.count() == len(chunks_df), (collection.count(), len(chunks_df))

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
from google import genai

load_dotenv(find_dotenv())
gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
MODEL = "gemini-3.8-flash"

PROMPT = """You answer questions about Nigerian banking regulation and compliance.

Rules you must follow:
1. Answer ONLY from the passages below. Do not use outside knowledge.
2. If the passages do not contain the answer, say exactly:
   "I could not find this in the available documents."
   Do not guess, and do not fill gaps from general knowledge.
3. Quote the specific rule or figure where there is one.
4. Refer to sources by their number, like [2].
5. A passage marked OCR was machine-read from a scanned document. If your
   answer relies on one, end with this line exactly as written, with no source
   number after it: "Note: this passage was read from a scan and may contain
   transcription errors - please verify figures and paragraph references
   against the source document."

PASSAGES
--------
{context}

QUESTION
--------
{question}

ANSWER"""


def build_context(hits):
    blocks = []
    for i, h in enumerate(hits, 1):
        tag = " [OCR]" if h.get("extraction") == "ocr" else ""
        ref = f" {h['ref_no']}" if h.get("ref_no") else ""
        blocks.append(
            f"[{i}] {h['regulator']}{ref} - {h['title']} (page {h['page']}){tag}\n{h['text']}"
        )
    return "\n\n".join(blocks)


def ask(question, k=5, n_sources=3):
    hits = search_hybrid(question, k=k)
    prompt = PROMPT.format(context=build_context(hits), question=question)
    response = gemini.models.generate_content(model=MODEL, contents=prompt)

    sources = [{
        "n": i,
        "regulator": h["regulator"],
        "ref_no": h.get("ref_no", ""),
        "title": h["title"],
        "page": h["page"],
        "url": h["url"],
        "ocr": h.get("extraction") == "ocr",
    } for i, h in enumerate(hits[:n_sources], 1)]

    return {"question": question, "answer": response.text.strip(), "sources": sources}


def print_answer(result):
    print(result["answer"])
    print("\nSources:")
    for s in result["sources"]:
        flag = " [OCR]" if s["ocr"] else ""
        ref = f" {s['ref_no']}" if s["ref_no"] else ""
        print(f"  [{s['n']}] {s['regulator']}{ref} - {s['title'][:60]}, p{s['page']}{flag}")
        print(f"      {s['url']}")


def demo(*args, **kwargs):
    """Run a narrative example, tolerating a spent daily quota so Run All survives."""
    try:
        print_answer(ask(*args, **kwargs))
    except Exception as exc:
        msg = str(exc)
        if "RESOURCE_EXHAUSTED" in msg or "429" in msg:
            print("[daily free-tier quota spent - live example skipped]")
        elif "UNAVAILABLE" in msg or "503" in msg:
            print("[model busy (503) - live example skipped]")
        else:
            raise


demo("What is the minimum paid-up share capital for a commercial bank with international authorisation?")

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print("=" * 70, "\n1. OUT OF SCOPE - must refuse\n")
demo("What are the minimum capital requirements for commercial banks in Kenya?")

print("\n" + "=" * 70, "\n2. OCR SOURCE - must carry the caveat\n")
demo("What is the penalty for a bank that fails to publish its audited accounts?")

print("\n" + "=" * 70, "\n3. VAGUE PHRASING - real user language\n")
demo("My bank wants to start doing business abroad. What do we need?")

1. OUT OF SCOPE - must refuse



ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [6]:
import json, hashlib
from pathlib import Path
from google.genai import errors as genai_errors

CACHE_PATH = Path("../data/answer_cache.json")
_cache = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}


def _key(model, question):
    return hashlib.sha1(f"{model}||{question}".encode()).hexdigest()


def ask(question, k=5, n_sources=3, model=None, use_cache=True):
    model = model or MODEL
    hits = search_hybrid(question, k=k)

    sources = [{
        "n": i, "regulator": h["regulator"], "ref_no": h.get("ref_no", ""),
        "title": h["title"], "page": h["page"], "url": h["url"],
        "ocr": h.get("extraction") == "ocr",
    } for i, h in enumerate(hits[:n_sources], 1)]

    ck = _key(model, question)
    if use_cache and ck in _cache:
        return {"question": question, "answer": _cache[ck], "sources": sources, "cached": True}

    prompt = PROMPT.format(context=build_context(hits), question=question)
    try:
        response = gemini.models.generate_content(model=model, contents=prompt)
    except genai_errors.ClientError as exc:
        if "RESOURCE_EXHAUSTED" in str(exc) or "429" in str(exc):
            return {"question": question, "sources": sources, "cached": False, "quota_hit": True,
                    "answer": "[QUOTA EXCEEDED - question not answered]"}
        raise

    text = response.text.strip()
    _cache[ck] = text
    CACHE_PATH.write_text(json.dumps(_cache, ensure_ascii=False, indent=1), encoding="utf-8")
    return {"question": question, "answer": text, "sources": sources, "cached": False}

In [7]:
demo("What are the minimum capital requirements for commercial banks in Kenya?",
     model="gemini-3.5-flash-lite")

I could not find this in the available documents.

Sources:
  [1] CBN FPR/DIR/PUB/CIR/002/009 - Review of Minimum Capital Requirements for Commercial, Merch, p4
      https://www.cbn.gov.ng/OUT/2024/CCD/RECAPITALIZATION_MARCH_2024.PDF
  [2] CBN FPR/DIR/PUB/CIR/002/009 - Review of Minimum Capital Requirements for Commercial, Merch, p5
      https://www.cbn.gov.ng/OUT/2024/CCD/RECAPITALIZATION_MARCH_2024.PDF
  [3] CBN FPR/DIR/PUB/CIR/002/009 - Review of Minimum Capital Requirements for Commercial, Merch, p5
      https://www.cbn.gov.ng/OUT/2024/CCD/RECAPITALIZATION_MARCH_2024.PDF


In [8]:
REFUSAL = "i could not find this in the available documents"


def print_answer(result):
    print(result["answer"])
    if REFUSAL in result["answer"].lower():
        print("\n(no sources - the question was not answered from the corpus)")
        return
    print("\nSources:")
    for s in result["sources"]:
        flag = " [OCR]" if s["ocr"] else ""
        ref = f" {s['ref_no']}" if s["ref_no"] else ""
        print(f"  [{s['n']}] {s['regulator']}{ref} - {s['title'][:60]}, p{s['page']}{flag}")
        print(f"      {s['url']}")

In [9]:
print("=" * 70, "\n2. OCR SOURCE - must carry the transcription caveat\n")
demo("What is the penalty for a bank that fails to publish its audited accounts?")

print("\n" + "=" * 70, "\n3. VAGUE PHRASING - how a real user types\n")
demo("My bank wants to start doing business abroad. What do we need?")

2. OCR SOURCE - must carry the transcription caveat



ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [10]:
def print_answer(result):
    print(result["answer"])
    if result.get("quota_hit"):
        print("\n(retrieval ran; these are the passages that would have been used)")
    elif REFUSAL in result["answer"].lower():
        print("\n(no sources - the question was not answered from the corpus)")
        return
    print("\nSources:")
    for s in result["sources"]:
        flag = " [OCR]" if s["ocr"] else ""
        ref = f" {s['ref_no']}" if s["ref_no"] else ""
        print(f"  [{s['n']}] {s['regulator']}{ref} - {s['title'][:60]}, p{s['page']}{flag}")
        print(f"      {s['url']}")

In [11]:
demo("What is the penalty for a bank that fails to publish its audited accounts?",
     model="gemini-3.5-flash-lite")

I could not find this in the available documents.

(no sources - the question was not answered from the corpus)


In [12]:
hits = search_hybrid("penalty for failing to publish audited accounts", k=10)
print("retrieved:")
for i, h in enumerate(hits, 1):
    print(f"  {i}. {h['title'][:38]} p{h['page']}")

mask = (chunks_df["text"].str.contains("not less than", case=False, na=False)
        & chunks_df["text"].str.contains("publish", case=False, na=False))
print("\nchunks mentioning both 'publish' and 'not less than':")
print(chunks_df.loc[mask, ["title", "page"]].head(10).to_string(index=False))

target = chunks_df[(chunks_df["title"].str.contains("BOFIA|Banks and Other", case=False, na=False))
                   & (chunks_df["page"] == 30)]
print(f"\nchunks on BOFIA p30: {len(target)}")
for _, r in target.iterrows():
    print("\n", r["text"][:400])

retrieved:
  1. Banks and Other Financial Institutions p29
  2. SEC Consolidated Rules and Regulations p73
  3. Banks and Other Financial Institutions p61
  4. New Rules and sundry amendments April  p23
  5. Executed Rules Dec 2024 p16
  6. SEC Consolidated Rules and Regulations p357
  7. Executed Rules Dec 2024 p15
  8. SEC Consolidated Rules and Regulations p192
  9. SEC Consolidated Rules and Regulations p25
  10. RE: GUIDELINES ON MANAGEMENT OF DORMAN p2

chunks mentioning both 'publish' and 'not less than':
                                          title  page
Banks and Other Financial Institutions Act 2020    29
Banks and Other Financial Institutions Act 2020    29
    SEC Consolidated Rules and Regulations 2013   211

chunks on BOFIA p30: 4

 tional daily newspapers
printed and circulating in Nigeria ;
(5) exhibit in a conspicuous position in each of its offices, branches and
website ; and
(c) forward to the Bank, copies of the bank's published Statement of
Financial Position an

In [13]:
from src.retrieval import chunks_df as _cdf

_ROWS = _cdf.to_dict("records")
_POS = {r["chunk_id"]: i for i, r in enumerate(_ROWS)}


def with_neighbours(hits, window=1):
    """Widen each hit to include the chunks either side, within the same document."""
    out = []
    for h in hits:
        i = _POS[h["chunk_id"]]
        lo, hi = max(0, i - window), min(len(_ROWS) - 1, i + window)
        parts = [_ROWS[j]["text"] for j in range(lo, hi + 1)
                 if _ROWS[j]["source_file"] == h["source_file"]]
        out.append({**h, "text": "\n".join(parts)})
    return out

In [14]:
import re


def cited_indices(answer):
    """Passage numbers the answer refers to, e.g. '... [1, 4]' -> [1, 4]."""
    idx = set()
    for g in re.findall(r"\[([\d,\s]+)\]", answer):
        idx.update(int(n) for n in re.findall(r"\d+", g))
    return sorted(idx)


def _src(h, i):
    return {"n": i, "regulator": h["regulator"], "ref_no": h.get("ref_no", ""),
            "title": h["title"], "page": h["page"], "url": h["url"],
            "ocr": h.get("extraction") == "ocr"}


def _sources_for(text, hits, n_sources):
    """Report what the answer cited. build_context numbers every passage sent to
    the model, so slicing the top n instead left citations like [4] unresolvable."""
    if REFUSAL in text.lower():
        return []
    cited = [i for i in cited_indices(text) if 1 <= i <= len(hits)]
    return [_src(hits[i - 1], i) for i in cited] or [
        _src(h, i) for i, h in enumerate(hits[:n_sources], 1)]


def ask(question, k=5, n_sources=3, model=None, use_cache=True, window=1):
    model = model or MODEL
    hits = with_neighbours(search_hybrid(question, k=k), window=window)
    ck = _key(f"{model}|w{window}", question)

    if use_cache and ck in _cache:
        text = _cache[ck]
        return {"question": question, "answer": text, "cached": True,
                "sources": _sources_for(text, hits, n_sources)}

    prompt = PROMPT.format(context=build_context(hits), question=question)
    try:
        response = gemini.models.generate_content(model=model, contents=prompt)
    except genai_errors.ClientError as exc:
        if "RESOURCE_EXHAUSTED" in str(exc) or "429" in str(exc):
            return {"question": question, "answer": "[QUOTA EXCEEDED - question not answered]",
                    "cached": False, "quota_hit": True,
                    "sources": [_src(h, i) for i, h in enumerate(hits[:n_sources], 1)]}
        raise

    text = response.text.strip()
    _cache[ck] = text
    CACHE_PATH.write_text(json.dumps(_cache, ensure_ascii=False, indent=1), encoding="utf-8")
    return {"question": question, "answer": text, "cached": False,
            "sources": _sources_for(text, hits, n_sources)}


print_answer(ask("What are the AML record keeping requirements?",
                 model="gemini-3.5-flash-lite"))

Based on the provided documents, the AML record keeping requirements include the following:

* **General Rule and Duration:** Records required by the ML/TF Acts and Regulations must, as a general rule, be kept for at least 5 years and made available to competent authorities on a timely basis [1, 2]. Similarly, Capital Market Operators must maintain all necessary records of transactions (domestic and international) for at least five years following the completion of the transaction or longer if requested by the SEC in specific cases, regardless of whether the account or business relationship is ongoing or terminated [3]. 
* **Customer Identification, Account Files, and Correspondence:** Records of identification data, the risk profile of each customer or beneficial owner, account files, CDD information, business correspondence, and the results of any analysis undertaken must be maintained for at least five years following the termination of an account or business relationship [3, 4].
* 

In [15]:
q_penalty = "What is the penalty for a bank that fails to publish its audited accounts?"
hits = with_neighbours(search_hybrid(q_penalty, k=5), window=1)
ctx = build_context(hits)

print("context length:", len(ctx))
print("contains '5,000,000':          ", "5,000,000" in ctx)
print("contains 'penalty of not less':", "penalty of not less" in ctx)
print("contains 'publish':            ", "publish" in ctx.lower())
print()
for i, h in enumerate(hits, 1):
    print(f"[{i}] {h['title'][:40]} p{h['page']} - {len(h['text'])} chars")

# All three are present, yet the model refuses. The obligation and its penalty
# are split mid-sentence by OCR ("of each such failure, liable to a penalty of
# not less than 5,000,000"), and the fragment never names what failed. This is a
# generation-side false refusal, not a retrieval or chunking gap - reranking and
# hybrid weighting will not touch it.
print("\n" + "=" * 70)
print(ctx[:2500])

context length: 11936
contains '5,000,000':           True
contains 'penalty of not less': True
contains 'publish':             True

[1] Banks and Other Financial Institutions A p55 - 2266 chars
[2] Banks and Other Financial Institutions A p29 - 2280 chars
[3] Banks and Other Financial Institutions A p34 - 2344 chars
[4] Banks and Other Financial Institutions A p36 - 2275 chars
[5] Banks and Other Financial Institutions A p11 - 2338 chars

[1] CBN BOFIA 2020 - Banks and Other Financial Institutions Act 2020 (page 55) [OCR]
imes the cumulative amount
collected ; or
e
(c) both such imprisonment or fine.
59.—(1) Any person who fails to comply with any of the conditions ofa
licence granted under section 58 is liable to a penalty of not less than
N2,000,000 and an additional fine of not less than N50,000 for each day during
which the contravention continues.
(2) Every person carrying on any financial business referred to in section
57 of this Act shall
*
(a) comply with the monetary policy

In [16]:
TEST_QUESTIONS_PATH = Path("../data/test_questions.json")

TEST_QUESTIONS = [
    {"id": 1, "question": "What is the minimum paid-up share capital for a commercial bank with international authorisation?",
     "expected_doc": "Review of Minimum Capital", "answer": "N500 billion"},
    {"id": 2, "question": "How long must a bank keep customer identification records?",
     "expected_doc": "Anti-Money Laundering/Combating the Financing", "answer": "5 years"},
    {"id": 3, "question": "What qualifications must members of a bank's board have?",
     "expected_doc": "Corporate Governance Guidelines",
     "answer": "Persons of proven integrity, knowledgeable in business and financial matters"},
    {"id": 4, "question": "What is the minimum capital adequacy ratio for a bank with international authorisation?",
     "expected_doc": "Guidelines on Regulatory Capital", "answer": "15%"},
    {"id": 5, "question": "What are the minimum capital requirements for commercial banks in Kenya?",
     "expected_doc": None, "answer": "Not in the corpus - the system should refuse"},
    {"id": 6, "question": "What does circular FPR/DIR/PUB/CIR/002/009 require?",
     "expected_doc": "Review of Minimum Capital", "answer": "The 2024 bank recapitalisation programme"},
    {"id": 8, "question": "What are the AML record keeping requirements?",
     "expected_doc": "Anti-Money Laundering/Combating the Financing", "answer": "5 years"},
    {"id": 9, "question": "What must a bank do when a customer is identified as a politically exposed person?",
     "expected_doc": "Politically Exposed", "answer": "Senior management approval, enhanced ongoing monitoring"},
    {"id": 10, "question": "How quickly must a bank report a cybersecurity incident to the CBN?",
     "expected_doc": "Cybersecurity", "answer": "?"},
    {"id": 11, "question": "When does an account become dormant and what happens to the balance?",
     "expected_doc": "DORMANT ACCOUNTS", "answer": "?"},
    {"id": 12, "question": "What liquidity ratio must a bank maintain?",
     "expected_doc": "Liquidity Risk Management", "answer": "?"},
    {"id": 13, "question": "What is the penalty for carrying on banking business without a licence?",
     "expected_doc": "Banks and Other Financial Institutions Act", "answer": "?"},
    {"id": 14, "question": "What identification documents does a new individual customer need to open an account?",
     "expected_doc": "Uniform Account Opening", "answer": "?"},
    {"id": 15, "question": "What is the NDIC's role when a bank fails?",
     "expected_doc": "Nigeria Deposit Insurance Corporation Act", "answer": "?"},
    {"id": 16, "question": "What capabilities must an automated anti-money laundering system have?",
     "expected_doc": "Baseline Standards", "answer": "?"},
    {"id": 17, "question": "How must a capital market operator handle a client complaint?",
     "expected_doc": "Complaints Management", "answer": "?"},
    {"id": 18, "question": "What are the custody requirements for digital assets?",
     "expected_doc": "Digital Assets", "answer": "?"},
]

TEST_QUESTIONS_PATH.write_text(json.dumps(TEST_QUESTIONS, indent=2), encoding="utf-8")
print("saved", len(TEST_QUESTIONS), "questions |",
      len([q for q in TEST_QUESTIONS if q["expected_doc"]]), "scored")

saved 17 questions | 16 scored


In [17]:
TEST_QUESTIONS = json.loads(Path("../data/test_questions.json").read_text(encoding="utf-8"))

In [ ]:
import time, pandas as pd

rows = []
for q in TEST_QUESTIONS:
    r = ask(q["question"], model="gemini-3.5-flash-lite")
    refused = REFUSAL in r["answer"].lower()
    rows.append({
        "id": q["id"],
        "question": q["question"],
        "expected_doc": q["expected_doc"] or "(not in corpus)",
        "expected_answer": q["answer"],
        "answer": r["answer"],
        "refused": refused,
        "quota_hit": r.get("quota_hit", False),
        "cached": r.get("cached", False),
        "top_source": r["sources"][0]["title"] if r["sources"] else "",
        "source_page": r["sources"][0]["page"] if r["sources"] else "",
        "n_cited": len(r["sources"]),
        "top_regulator": r["sources"][0]["regulator"] if r["sources"] else "",
        "all_sources": " | ".join(f'[{s["n"]}] {s["regulator"]} - {s["title"]} p{s["page"]}'
                                  for s in r["sources"]),
        "ocr_source": any(s["ocr"] for s in r["sources"]),
        "correct": "",          # you fill this in
        "notes": "",            # and this
    })
    if not r.get("cached"):
        time.sleep(1)

results = pd.DataFrame(rows)

# Grading is entered by hand in the CSV. Carry it across re-runs of this cell,
# otherwise a free cached re-run silently discards the part that took effort.
RESULTS_PATH = Path("../data/phase6_results.csv")
if RESULTS_PATH.exists():
    prev = pd.read_csv(RESULTS_PATH).set_index("id")
    for col in ("correct", "notes"):
        if col in prev.columns:
            results[col] = results["id"].map(prev[col]).fillna("")
    kept = int((results["correct"].astype(str).str.strip() != "").sum())
    print(f"carried over {kept} graded rows")

results.to_csv(RESULTS_PATH, index=False)

print("answered:", int((~results["refused"] & ~results["quota_hit"]).sum()),
      "| refused:", int(results["refused"].sum()),
      "| quota:", int(results["quota_hit"].sum()))
print()
print(results[["id", "refused", "n_cited", "top_source"]].to_string(index=False))

In [19]:
import textwrap


def doc_matched(row):
    if not row["expected_doc"] or row["expected_doc"].startswith("("):
        return None
    return row["expected_doc"].lower() in str(row["top_source"]).lower()


results["doc_match"] = results.apply(doc_matched, axis=1)
mismatched = results[results["doc_match"] == False]

for _, row in mismatched.iterrows():
    r = ask(row["question"], model="gemini-3.5-flash-lite")   # cached
    print("=" * 70)
    print(f'[{row["id"]}] {row["question"]}')
    print(f'  expected: {row["expected_doc"]}')
    print(f'  cited:    ' + " | ".join(f'[{s["n"]}] {s["title"][:40]} (p{s["page"]})'
                                       for s in r["sources"]))
    print()
    print(textwrap.fill(r["answer"], 100, initial_indent="  ", subsequent_indent="  "))
    print()

[3] What qualifications must members of a bank's board have?
  expected: Corporate Governance Guidelines
  cited:    [4] SEC Consolidated Rules and Regulations 2 (p412) | [5] Corporate Governance Guidelines for Comm (p7)

  Based on the available documents, members of a bank's board must meet the following qualification
  requirements:  * Appointees to the Board must have a track record covering both integrity and past
  performance, in accordance with extant Guidelines on competency and fit and proper persons for the
  Nigerian banking industry [5]. * Members should be individuals with "upright personal
  characteristics, relevant core competences and entrepreneurial spirit", have a "record of tangible
  achievement", and be "knowledgeable in Board matters" [4.4]. Additionally, they should "possess a
  sense of accountability and integrity and be committed to the task of good corporate governance"
  [4].

[8] What are the AML record keeping requirements?
  expected: Anti-Money Launder

In [20]:
for p in sorted(Path("../data").iterdir()):
    print(f"{p.name:45s} {p.stat().st_size/1e6:8.2f} MB")

print()
print(len(chunks_df), "chunks |", chunks_df["title"].nunique(), "documents")

answer_cache.json                                 0.02 MB
chunks.jsonl                                      4.58 MB
phase6_results.csv                                0.02 MB
processed                                         0.01 MB
raw                                               0.01 MB
retrieval_comparison.csv                          0.00 MB
retrieval_sweep.csv                               0.00 MB
sources.csv                                       0.01 MB
test_questions.json                               0.00 MB

3828 chunks | 24 documents


In [21]:
hits = chunks_df[chunks_df["title"].str.contains("omplaint", case=False, na=False)]
print(hits.groupby("title").size() if len(hits) else "not in the index")

title
Rules Relating to the Complaints Management Framework of the Nigerian Capital Market - February 12, 2015    16
dtype: int64


In [22]:
for t in sorted(chunks_df["title"].unique()):
    print(t[:100])

Additional Know Your Customer Requirement in Respect of Non-Profit Organizations
Anti-Money Laundering/Combating the Financing of Terrorism (AML/CFT) Policy and Procedure Manual
Banks and Other Financial Institutions Act 2020
Central Bank of Nigeria Risk-Based Cybersecurity Framework and Guidelines for Deposit Money Banks an
Circular to All Banks and Other Financial Institutions: Uniform Account Opening Forms and Minimum In
Corporate Governance Guidelines for Commercial Banks, Merchant, Non-Interest and Payment Service Ban
Executed Rules Dec 2024
Guidance Note on Anti-Money Laundering and Combating the Financing of Terrorism (AML/CFT) Regulation
Guidance Note on Politically Exposed Persons (PEP)
Guidelines for Licensing of Banks and Other Financial Institutions in Nigeria on Anti-Money Launderi
Guidelines on Liquidity Risk Management and Internal Liquidity Adequacy Assessment Process (ILAAP)
Guidelines on Regulatory Capital
Issuance of Baseline Standards for Automated Anti-Money Launde

In [23]:
q17 = "How must a capital market operator handle a client complaint?"

for i, h in enumerate(search_hybrid(q17, k=20), 1):
    print(f'{i:3d}  {h["title"][:60]} {"<<<" if "omplaint" in h["title"] else ""}')

  1  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
  2  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
  3  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
  4  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
  5  SEC Consolidated Rules and Regulations 2013 
  6  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
  7  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
  8  SEC Consolidated Rules and Regulations 2013 
  9  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
 10  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
 11  SEC Consolidated Rules and Regulations 2013 
 12  SEC Consolidated Rules and Regulations 2013 
 13  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
 14  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
 15  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
 16  SEC Capital Market Operators AML/CFT/CPF Regulations 2022 
 17  SEC Capital Market Operator

In [3]:
# Phase 7 check - the pieces above now live in src/rag.py.
# Self-contained on purpose: run it on a fresh kernel, on its own, and it
# exercises the module rather than the definitions already in memory.
import sys
sys.path.insert(0, "..")

from src.rag import ask, print_answer

print_answer(ask("What are the AML record keeping requirements?",
                 model="gemini-3.5-flash-lite"))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Based on the provided documents, the AML record keeping requirements include the following:

* **General Rule and Duration:** Records required by the ML/TF Acts and Regulations must, as a general rule, be kept for at least 5 years and made available to competent authorities on a timely basis [1, 2]. Similarly, Capital Market Operators must maintain all necessary records of transactions (domestic and international) for at least five years following the completion of the transaction or longer if requested by the SEC in specific cases, regardless of whether the account or business relationship is ongoing or terminated [3]. 
* **Customer Identification, Account Files, and Correspondence:** Records of identification data, the risk profile of each customer or beneficial owner, account files, CDD information, business correspondence, and the results of any analysis undertaken must be maintained for at least five years following the termination of an account or business relationship [3, 4].
* 

In [2]:
import sys
sys.path.insert(0, "..")

from src.rag import ask, print_answer

print_answer(ask("How must a capital market operator handle a client complaint?",
                 model="gemini-3.5-flash-lite", use_cache=False))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Based on the provided documents, a capital market operator must handle client complaints in the following ways:

* All Capital Market Operators and listed Public Companies are required to establish a "clearly defined Complaints Management policy to handle and resolve complaints from their clients" [1, 3].
* The complaints management policy must deal with complaints against operators by clients, other operators, shareholders/public companies, and investors [1]. 
* The policy must be defined and endorsed by the company’s or firm’s senior management, who are also responsible for its implementation and for monitoring compliance [1].
* Competent authorities must ensure that companies and firms have a complaints management function that enables complaints to be investigated fairly and allows potential conflicts of interest to be identified and mitigated [1].

Note: this passage was read from a scan and may contain transcription errors - please verify figures and paragraph references against 